# MTAM reproduction — ZuCo sentiment analysis

This is the single Colab notebook for the `reproduction` branch of `parmisbathayan/EEG_Language_Alignment`. Data and run outputs remain in Google Drive.

**Current step: v16b, the single bounded follow-up to the completed feature-construction audit.** It tests four remaining questions only: sentence `mean_*_sec` row averaging, duration-weighted FFD/TRT pooling, similarity after the exact released first-104/per-band Z-score, and the stored 105th value/channel.

This step does not train a model, inspect raw voltage, search for new EEG features, or use labels/test performance to select a representation. After this one audit we stop pooling experiments and move to the frozen released 832-feature MLP baseline.

No GPU or additional package installation is required for v16b.

## 1. Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OG_ZUCO_SR_DIR = '/content/drive/MyDrive/Thesis/Data/zuco_og_raw'
RESULTS_ROOT = '/content/drive/MyDrive/Thesis/Results/reproduce_EEG_Language_Alignment'
FORK_URL = 'https://github.com/parmisbathayan/EEG_Language_Alignment.git'
BRANCH = 'reproduction'

import os
assert os.path.isdir(OG_ZUCO_SR_DIR), f'SR .mat folder not found: {OG_ZUCO_SR_DIR}'
mat_files = sorted(name for name in os.listdir(OG_ZUCO_SR_DIR) if name.endswith('.mat'))
assert mat_files, f'No .mat files found in {OG_ZUCO_SR_DIR}'
os.makedirs(RESULTS_ROOT, exist_ok=True)
print('Drive mounted. SR files:', mat_files)

## 2. Clone the current reproduction branch

Re-running this cell refreshes the Colab copy from GitHub. The Drive data is not touched.

In [ ]:
%cd /content
!rm -rf /content/EEG_Language_Alignment
!git clone --branch {BRANCH} {FORK_URL}
%cd /content/EEG_Language_Alignment

print('Checked-out commit:')
!git rev-parse HEAD
print('\nChanges relative to upstream:')
!git remote add upstream https://github.com/Jason-Qiu/EEG_Language_Alignment.git 2>/dev/null; git fetch -q upstream
!git log --oneline upstream/main..{BRANCH}

## 3. Link the original ZuCo `.mat` files

The audit reads the files in place through symlinks; it does not duplicate or modify the Drive data.

In [ ]:
%cd /content/EEG_Language_Alignment
import os

os.makedirs('data/SR', exist_ok=True)
for filename in mat_files:
    destination = os.path.join('data/SR', filename)
    if not os.path.lexists(destination):
        os.symlink(os.path.join(OG_ZUCO_SR_DIR, filename), destination)

linked_files = sorted(name for name in os.listdir('data/SR') if name.endswith('.mat'))
assert linked_files == mat_files, (linked_files, mat_files)
print('Linked SR files:', linked_files)

import numpy as np, scipy
print('NumPy:', np.__version__, '| SciPy:', scipy.__version__)

## 4. Run v16b — bounded construction follow-up

This produces one timestamped log and one JSON in Drive under `v16b_featureConstructionFollowup`. It compares the five predeclared candidates both before and after the released 104-channel Z-score, records exactly how many participants contribute to each candidate, and audits the 105th stored value plus any channel-label-like metadata.

The two post-Z-score targets are saved separately: the actual released all-participant sentence vector, and a diagnostic target averaged over only the participants available to that candidate. They must not be confused.

In [ ]:
import datetime, os, subprocess

audit_timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
audit_dir = os.path.join(RESULTS_ROOT, 'v16b_featureConstructionFollowup')
log_dir = os.path.join(audit_dir, 'logs')
os.makedirs(log_dir, exist_ok=True)

audit_json = os.path.join(audit_dir, f'feature_construction_followup_{audit_timestamp}.json')
audit_log = os.path.join(log_dir, f'feature_construction_followup_{audit_timestamp}.txt')
audit_command = [
    'python', '-u', 'audit_feature_construction_followup.py',
    '--eeg_dir', 'data/SR',
    '--output_json', audit_json,
]

print('Running v16b bounded feature-construction follow-up')
print('JSON:', audit_json)
print('Log:', audit_log)
print('-' * 60)
with open(audit_log, 'w') as log_file:
    log_file.write('Command: ' + ' '.join(audit_command) + '\n' + '-' * 60 + '\n')
    process = subprocess.Popen(
        audit_command, cwd='/content/EEG_Language_Alignment',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'v16b audit failed with exit code {return_code}. See {audit_log}')
print('-' * 60)
print('v16b complete.')

## 5. Show the compact result summary

Run these cells after section 4. The complete machine-readable evidence remains in the JSON. `exact-ish vectors` means individual sentence-band vectors whose maximum electrode error is at most `0.001`; it does not mean full 832-feature sentence examples.

In [ ]:
import json

with open(audit_json) as audit_file:
    audit = json.load(audit_file)

print('Commit:', audit['provenance']['git_commit'])
print('Included participant files:', len(audit['source']['included_files']))
print('Valid participant-sentence targets:', audit['source']['valid_released_participant_sentence_records'])
print('Excluded:', audit['source']['excluded_files'])

print('\nComplete participant-sentence candidates:')
for name, count in audit['coverage']['candidate_complete_participant_sentence_records'].items():
    contributors = audit['coverage']['candidate_contributors_per_sentence'][name]
    print(f"  {name}: records={count}, contributors/sentence={contributors}")

def show_aggregate(title, aggregate):
    print('\n' + title)
    for name, result in aggregate.items():
        print(
            f"  {name}: rel_RMSE={result['mean_relative_rmse']:.6g}, "
            f"corr={result['mean_flattened_correlation']}, "
            f"exact-ish vectors={result['total_vectors_with_max_abs_error_at_most_0_001']}"
        )

show_aggregate(
    'Participant-level raw comparison (first 104):',
    audit['participant_level_raw_comparison_first_104']['aggregate_across_bands'],
)
show_aggregate(
    'Final candidate vs ACTUAL released all-participant target after Z-score:',
    audit['final_sentence_representation_after_released_zscore']
         ['candidate_vs_released_all_subject_target']['aggregate_across_bands'],
)
show_aggregate(
    'Diagnostic candidate vs SAME-CONTRIBUTOR target after Z-score:',
    audit['final_sentence_representation_after_released_zscore']
         ['candidate_vs_target_using_same_contributing_subjects']['aggregate_across_bands'],
)

print('\n105th stored value by band:')
for band, result in audit['stored_105th_value']['per_band'].items():
    summary = result['finite_value_summary']
    print(
        f"  {band}: finite={result['finite_105th_values']}, "
        f"NaN={result['nan_105th_values']}, "
        f"std={None if summary is None else summary['standard_deviation']}"
    )
metadata_hits = [
    (filename, metadata['label_like_metadata_candidates'])
    for filename, metadata in audit['stored_105th_value']['metadata_snapshots'].items()
    if metadata['label_like_metadata_candidates']
]
print('Channel-label-like metadata files:', len(metadata_hits))
if metadata_hits:
    filename, found = metadata_hits[0]
    print('  First file:', filename)
    for path, details in found.items():
        print(' ', path, details['snapshot'], details['string_values'])

print('\nSaved JSON:', audit_json)
print('Saved log:', audit_log)

In [ ]:
show_aggregate(
    'Effect of including the 105th value in each band Z-score:',
    audit['stored_105th_value']
         ['effect_of_including_105th_value_on_first_104_zscores']['aggregate'],
)

---
### After v16b
Tell Codex that v16b finished. We will read the Drive JSON, record the result and remaining ambiguity, and then stop feature-pooling audits. The next planned run is v17: the frozen multi-seed MLP using the exact released 832-feature sentence representation. Do not add TRT, raw EEG, or a 105-channel model based on this audit.